# Project Document Scanner

Code

In [ ]:
#!/usr/bin/env python3
"""
document_detector_algorithmic.py

Algorithmic-only robust document finder + tracker.

Keys:
  q  - quit
  d  - toggle debug overlays (shows masks / candidates)
  s  - save current warped crop to disk

Tweak parameters in the CONFIG section.
"""
import cv2
import numpy as np
import os
import threading
import time
from collections import deque

# -----------------------
# CONFIG (tweak here)
# -----------------------
CAM_IDX = 1
TARGET_W, TARGET_H = 1280, 720       # camera capture size
DETECT_SCALE = 0.5                   # detection runs on small frame (speed)
MIN_CONTOUR_AREA = int(0.004 * TARGET_W * TARGET_H)
MAX_FULL_FRAME_RATIO = 0.96
SMOOTH_ALPHA = 0.28
SIZE_SMOOTH_ALPHA = 0.12
SWITCH_FRAMES = 4
IOU_IMMEDIATE_SWITCH = 0.70
AREA_RATIO_IMMEDIATE = 1.5
LK_MAX_AGE = 10

# MSER / morphology
MSER_DELTA = 5
MSER_MIN_AREA = 60
MSER_MAX_AREA = int(0.9 * TARGET_W * TARGET_H)

# KLT params
LK_PARAMS = dict(winSize=(21,21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))

# Enhancement worker toggles (keeps UI responsive)
USE_ENHANCER = True
UPSCALE_FACTOR = 1.5
UNSHARP_WEIGHT = 1.6
UNSHARP_BLUR_WEIGHT = -0.6
UNSHARP_SIGMA = 1.0
CLAHE_CLIP = 3.0
CLAHE_TILE = (8,8)

# Display
DISP_FACTOR = 3
DISP_W = TARGET_W // DISP_FACTOR
DISP_H = TARGET_H // DISP_FACTOR

OUT_DIR = "saved_warps"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# Camera & windows
# -----------------------
cap = cv2.VideoCapture(CAM_IDX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, TARGET_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, TARGET_H)

cv2.namedWindow("WorkFlow", cv2.WINDOW_NORMAL)
cv2.resizeWindow("WorkFlow", DISP_W, DISP_H*2)
cv2.namedWindow("Detection Preview", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Detection Preview", DISP_W, DISP_H)

# -----------------------
# Shared state
# -----------------------
prev_corners = None
selected_corners = None
selected_area = None
selected_center = None
selected_age = 0
candidate_corners = None
candidate_age = 0
prev_warp = None
prev_warp_size = None

# trackers
bbox_tracker = None   # MOSSE
tracker_bbox = None
prev_gray = None
lk_age = 0
tracker_points = None

# enhancer worker state
_worker_lock = threading.Lock()
_worker_input = None
_worker_stop = False
last_enhanced = None
last_enhanced_corners = None

debug_mode = False

# -----------------------
# Utility functions
# -----------------------
def reorder(pts):
    pts = pts.reshape(4,2).astype(np.float32)
    y_sorted = pts[np.argsort(pts[:,1])]
    top = y_sorted[:2]; bottom = y_sorted[2:]
    top = top[np.argsort(top[:,0])]
    bottom = bottom[np.argsort(bottom[:,0])]
    return np.array([top[0], top[1], bottom[0], bottom[1]], dtype=np.float32)

def rect_iou(boxA, boxB):
    xA1, yA1 = boxA[:,0].min(), boxA[:,1].min()
    xA2, yA2 = boxA[:,0].max(), boxA[:,1].max()
    xB1, yB1 = boxB[:,0].min(), boxB[:,1].min()
    xB2, yB2 = boxB[:,0].max(), boxB[:,1].max()
    inter_x1 = max(xA1, xB1); inter_y1 = max(yA1, yB1)
    inter_x2 = min(xA2, xB2); inter_y2 = min(yA2, yB2)
    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
        return 0.0
    inter = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    areaA = max(0.0, (xA2 - xA1) * (yA2 - yA1))
    areaB = max(0.0, (xB2 - xB1) * (yB2 - yB1))
    union = areaA + areaB - inter
    if union <= 0:
        return 0.0
    return inter / union

def get_area_of_poly(box):
    return abs(cv2.contourArea(box.reshape(4,1,2).astype(np.int32)))

def approx_quad(cnt):
    peri = cv2.arcLength(cnt, True)
    for eps_factor in (0.02, 0.015, 0.01, 0.025):
        approx = cv2.approxPolyDP(cnt, eps_factor * peri, True)
        if approx is None:
            continue
        if len(approx) == 4:
            return approx.reshape(4,1,2).astype(np.int32)
        if len(approx) > 4:
            hull = cv2.convexHull(approx)
            peri2 = cv2.arcLength(hull, True)
            approx2 = cv2.approxPolyDP(hull, 0.02 * peri2, True)
            if approx2 is not None and len(approx2) == 4:
                return approx2.reshape(4,1,2).astype(np.int32)
    return None

def smooth_points(prev_pts, new_pts, alpha=SMOOTH_ALPHA):
    if prev_pts is None:
        return new_pts.copy().astype(np.float32)
    return (alpha * new_pts + (1.0 - alpha) * prev_pts).astype(np.float32)

def getWarp(img, corners, smooth_size=None):
    pts1 = reorder(corners)
    widthA = np.linalg.norm(pts1[0] - pts1[1])
    widthB = np.linalg.norm(pts1[2] - pts1[3])
    maxWidth = max(int(widthA), int(widthB), 1)
    heightA = np.linalg.norm(pts1[0] - pts1[2])
    heightB = np.linalg.norm(pts1[1] - pts1[3])
    maxHeight = max(int(heightA), int(heightB), 1)
    if smooth_size is not None:
        maxWidth = max(1, int(smooth_size[0])); maxHeight = max(1, int(smooth_size[1]))
    pts2 = np.float32([[0,0],[maxWidth-1,0],[0,maxHeight-1],[maxWidth-1,maxHeight-1]])
    matrix = cv2.getPerspectiveTransform(pts1, pts2)
    warped = cv2.warpPerspective(img, matrix, (maxWidth, maxHeight), flags=cv2.INTER_CUBIC)
    return warped, (maxWidth, maxHeight)

def validate_quad(pts, img_w, img_h):
    # basic geometric checks: area, aspect, rectangularity approximated
    if pts is None:
        return False
    area = get_area_of_poly(pts)
    if area < MIN_CONTOUR_AREA or area > img_w * img_h * MAX_FULL_FRAME_RATIO:
        return False
    rect = cv2.minAreaRect(pts.reshape(4,1,2).astype(np.int32))
    w,h = rect[1]
    if min(w,h) < 5:
        return False
    ar = max(w,h) / (min(w,h) + 1e-5)
    if not (0.25 < ar < 8.0):
        return False
    return True

# -----------------------
# Enhancement worker
# -----------------------
def enhance_image_basic(img):
    out = img.copy()
    # upscale modestly (keeps speed)
    out = cv2.resize(out, None, fx=UPSCALE_FACTOR, fy=UPSCALE_FACTOR, interpolation=cv2.INTER_CUBIC)
    blurred = cv2.GaussianBlur(out, (0,0), sigmaX=UNSHARP_SIGMA)
    sharpened = cv2.addWeighted(out, UNSHARP_WEIGHT, blurred, UNSHARP_BLUR_WEIGHT, 0)
    lab = cv2.cvtColor(sharpened, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    cl = clahe.apply(l)
    merged = cv2.merge((cl,a,b))
    final = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    return final

def worker_loop():
    global _worker_input, _worker_stop, last_enhanced, last_enhanced_corners
    print("Enhancer worker started")
    while not _worker_stop:
        item = None
        with _worker_lock:
            if _worker_input is not None:
                item = _worker_input
                _worker_input = None
        if item is None:
            time.sleep(0.02)
            continue
        try:
            cropped_img, corners_snapshot = item
            enhanced = enhance_image_basic(cropped_img)
            with _worker_lock:
                last_enhanced = enhanced
                last_enhanced_corners = corners_snapshot.copy() if corners_snapshot is not None else None
        except Exception as e:
            print("Worker exception:", e)
    print("Enhancer worker stopping")

if USE_ENHANCER:
    worker = threading.Thread(target=worker_loop, daemon=True)
    worker.start()

# -----------------------
# Detector: multi-method on small frame (fast)
# -----------------------
def detect_candidates(frame_full):
    """
    Runs on downscaled copy for speed. Returns list of candidate quads in full-frame coords.
    """
    Hf, Wf = frame_full.shape[:2]
    small = cv2.resize(frame_full, (int(Wf*DETECT_SCALE), int(Hf*DETECT_SCALE)))
    H, W = small.shape[:2]
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    # CLAHE to help shadows
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    g = clahe.apply(gray)

    candidates = []

    # --- method 1: adaptive threshold + morphology ---
    th = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, 51, 9)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7,7))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
    thr_cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in thr_cnts:
        area = cv2.contourArea(cnt)
        if area < max(8, MIN_CONTOUR_AREA * (DETECT_SCALE**2)):
            continue
        if area / (H*W) > MAX_FULL_FRAME_RATIO:
            continue
        q = approx_quad(cnt)
        if q is None:
            hull = cv2.convexHull(cnt)
            rect = cv2.minAreaRect(hull)
            pts = cv2.boxPoints(rect).astype(np.int32)
            q = pts.reshape(4,1,2)
        # scale to full frame
        q_full = (q.astype(np.float32) / DETECT_SCALE).astype(np.int32)
        candidates.append(q_full)

    # --- method 2: canny edges ---
    edges = cv2.Canny(g, 50, 150)
    edges = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)
    e_cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in e_cnts:
        area = cv2.contourArea(cnt)
        if area < max(8, MIN_CONTOUR_AREA * (DETECT_SCALE**2)):
            continue
        q = approx_quad(cnt)
        if q is not None:
            q_full = (q.astype(np.float32) / DETECT_SCALE).astype(np.int32)
            candidates.append(q_full)

    # --- method 3: MSER regions (good for text regions) ---
    try:
        mser = cv2.MSER_create(_delta=MSER_DELTA, _min_area=MSER_MIN_AREA, _max_area=MSER_MAX_AREA)
        regions, _ = mser.detectRegions(g)
        for r in regions:
            hull = cv2.convexHull(r.reshape(-1,1,2))
            area = cv2.contourArea(hull)
            if area < max(8, MIN_CONTOUR_AREA * (DETECT_SCALE**2)):
                continue
            rect = cv2.minAreaRect(hull)
            pts = cv2.boxPoints(rect).astype(np.int32)
            q_full = (pts.astype(np.float32) / DETECT_SCALE).astype(np.int32)
            candidates.append(q_full)
    except Exception:
        pass

    # --- post-process: filter / deduplicate / score ---
    filtered = []
    seen = []
    for q in candidates:
        # clean shape
        try:
            qf = q.reshape(4,2).astype(np.float32)
        except Exception:
            continue
        if not validate_quad(qf, Wf, Hf):
            continue
        # dedupe by IoU with already seen
        dup = False
        for s in seen:
            if rect_iou(qf, s) > 0.85:
                dup = True; break
        if dup:
            continue
        seen.append(qf)
        filtered.append(qf)

    # compute scores and sort
    scored = []
    cx_img, cy_img = Wf/2.0, Hf/2.0
    for q in filtered:
        area = get_area_of_poly(q)
        hull = cv2.convexHull(q.reshape(-1,1,2).astype(np.int32))
        hull_area = max(cv2.contourArea(hull), 1.0)
        rect = cv2.minAreaRect(hull)
        rect_area = max(rect[1][0] * rect[1][1], 1.0)
        rectangularity = hull_area / rect_area
        extent = area / hull_area
        M = cv2.moments(hull)
        if M["m00"] != 0:
            cx = M["m10"]/M["m00"]; cy = M["m01"]/M["m00"]
        else:
            cx, cy = cx_img, cy_img
        center_dist = np.hypot(cx - cx_img, cy - cy_img) / np.hypot(cx_img, cy_img)
        center_score = 1.0 - center_dist
        size_score = min(area / (Wf*Hf) * 6.0, 1.0)
        ar = max(rect[1][0], rect[1][1]) / (min(rect[1][0], rect[1][1]) + 1e-5)
        ar_score = 1.0 if 0.25 < ar < 8.0 else 0.3
        score = (0.45 * rectangularity + 0.20 * extent + 0.10 * ar_score +
                 0.15 * size_score + 0.1 * center_score)
        scored.append((score, q))
    scored.sort(key=lambda x: x[0], reverse=True)
    # return up to top 6 candidates
    top_quads = [q for _,q in scored[:6]]
    return top_quads, th, edges

# -----------------------
# Main loop
# -----------------------
save_count = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("camera read failed")
            time.sleep(0.3)
            continue
        frame = cv2.resize(frame, (TARGET_W, TARGET_H))
        preview = frame.copy()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # run detector less expensively: detect candidates on small image
        candidates, mask_small, edges_small = detect_candidates(frame)

        # Select best candidate by comparing to currently selected
        best_det = None
        best_score = -1.0
        for q in candidates:
            area = get_area_of_poly(q)
            score = area  # simple primary metric: larger better (we already scored in detect)
            # compute IoU vs selected -> prefer different if much larger
            if selected_corners is not None:
                iou = rect_iou(q, selected_corners)
                if iou > IOU_IMMEDIATE_SWITCH:
                    best_det = q
                    best_score = 1e9
                    break
            if score > best_score:
                best_score = score
                best_det = q

        # Candidate acceptance logic & stability buffering
        if best_det is not None:
            det_pts = reorder(best_det).astype(np.float32)
            det_area = get_area_of_poly(det_pts)
            if selected_corners is None:
                selected_corners = det_pts.copy()
                selected_area = det_area
                selected_center = np.mean(selected_corners, axis=0)
                candidate_age = 0
                candidate_corners = None
                selected_age = 0
                lk_age = 0
                # init trackers: MOSSE bbox + KLT corners
                x1 = int(selected_corners[:,0].min()); y1 = int(selected_corners[:,1].min())
                x2 = int(selected_corners[:,0].max()); y2 = int(selected_corners[:,1].max())
                tracker_bbox = (x1, y1, x2-x1, y2-y1)
                try:
                    bbox_tracker = cv2.TrackerMOSSE_create()
                    bbox_tracker.init(frame, tracker_bbox)
                except Exception:
                    bbox_tracker = None
                tracker_points = selected_corners.reshape(-1,1,2).astype(np.float32)
                prev_gray = gray.copy()
            else:
                iou = rect_iou(det_pts, selected_corners)
                if iou >= IOU_IMMEDIATE_SWITCH or (det_area >= (selected_area or 1.0) * AREA_RATIO_IMMEDIATE):
                    # immediate accept
                    selected_corners = det_pts.copy()
                    selected_area = det_area
                    selected_center = np.mean(selected_corners, axis=0)
                    selected_age = 0
                    candidate_age = 0
                    lk_age = 0
                    tracker_points = selected_corners.reshape(-1,1,2).astype(np.float32)
                    prev_gray = gray.copy()
                    # reset MOSSE
                    x1 = int(selected_corners[:,0].min()); y1 = int(selected_corners[:,1].min())
                    x2 = int(selected_corners[:,0].max()); y2 = int(selected_corners[:,1].max())
                    tracker_bbox = (x1, y1, x2-x1, y2-y1)
                    try:
                        bbox_tracker = cv2.TrackerMOSSE_create()
                        bbox_tracker.init(frame, tracker_bbox)
                    except Exception:
                        bbox_tracker = None
                else:
                    # candidate buffering
                    if candidate_corners is None or np.max(np.linalg.norm(candidate_corners - det_pts, axis=1)) > 8.0:
                        candidate_corners = det_pts.copy()
                        candidate_age = 1
                    else:
                        candidate_age += 1
                        if candidate_age >= SWITCH_FRAMES:
                            selected_corners = det_pts.copy()
                            selected_area = det_area
                            selected_center = np.mean(selected_corners, axis=0)
                            selected_age = 0
                            candidate_age = 0
                            lk_age = 0
                            tracker_points = selected_corners.reshape(-1,1,2).astype(np.float32)
                            prev_gray = gray.copy()
                            # reset MOSSE
                            x1 = int(selected_corners[:,0].min()); y1 = int(selected_corners[:,1].min())
                            x2 = int(selected_corners[:,0].max()); y2 = int(selected_corners[:,1].max())
                            tracker_bbox = (x1, y1, x2-x1, y2-y1)
                            try:
                                bbox_tracker = cv2.TrackerMOSSE_create()
                                bbox_tracker.init(frame, tracker_bbox)
                            except Exception:
                                bbox_tracker = None
        else:
            # No fresh detection: try trackers
            tracked_ok = False
            if bbox_tracker is not None:
                ok, bb = bbox_tracker.update(frame)
                if ok:
                    x,y,w,h = map(int, bb)
                    # infer corners from bbox as fallback
                    bbox_corners = np.array([[x,y],[x+w,y],[x,y+h],[x+w,y+h]], dtype=np.float32)
                    if selected_corners is None:
                        selected_corners = bbox_corners.copy()
                        tracker_points = selected_corners.reshape(-1,1,2).astype(np.float32)
                        prev_gray = gray.copy()
                        tracked_ok = True
                    else:
                        # refine corners via KLT
                        tracked_ok = True
                        # don't immediately replace selected_corners; feed into KLT below
                else:
                    bbox_tracker = None

            # KLT corner tracking if possible
            if selected_corners is not None and prev_gray is not None and tracker_points is not None:
                p0 = tracker_points
                p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, p0, None, **LK_PARAMS)
                if p1 is not None and st is not None and st.sum() >= 2:
                    good = (st.reshape(-1) == 1)
                    if np.count_nonzero(good) >= 2:
                        p1_full = selected_corners.copy()
                        idx = 0
                        for i in range(4):
                            if good[i]:
                                p1_full[i] = p1[idx].reshape(2)
                            idx += 1
                        p1_full[:,0] = np.clip(p1_full[:,0], 0, frame.shape[1]-1)
                        p1_full[:,1] = np.clip(p1_full[:,1], 0, frame.shape[0]-1)
                        selected_corners = smooth_points(selected_corners, p1_full, alpha=SMOOTH_ALPHA)
                        selected_age += 1
                        lk_age += 1
                        tracker_points = selected_corners.reshape(-1,1,2).astype(np.float32)
                        prev_gray = gray.copy()
                    else:
                        lk_age += 1
                else:
                    lk_age += 1
            else:
                lk_age += 1

        # If we have selected corners, visualize and warp
        if selected_corners is not None:
            prev_corners = smooth_points(prev_corners, selected_corners, alpha=SMOOTH_ALPHA)
            preview_viz = preview.copy()
            pts = prev_corners.reshape(4,2).astype(int)
            cv2.polylines(preview_viz, [pts[[0,1,3,2]]], True, (0,255,0), 2)
            for i,p in enumerate(pts):
                cv2.circle(preview_viz, tuple(p), 5, (0,255,0), -1)
                cv2.putText(preview_viz, str(i), tuple(p+np.array([6,-6])), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            cv2.imshow("Detection Preview", cv2.resize(preview_viz, (DISP_W, DISP_H)))

            # warp size smoothing
            pts1 = reorder(prev_corners)
            widthA = np.linalg.norm(pts1[0] - pts1[1]); widthB = np.linalg.norm(pts1[2] - pts1[3])
            rawW = max(int(max(widthA, widthB)), 1)
            heightA = np.linalg.norm(pts1[0] - pts1[2]); heightB = np.linalg.norm(pts1[1] - pts1[3])
            rawH = max(int(max(heightA, heightB)), 1)
            if prev_warp_size is None:
                prev_warp_size = np.array([rawW, rawH], dtype=np.float32)
            else:
                prev_warp_size = (SIZE_SMOOTH_ALPHA * np.array([rawW, rawH], dtype=np.float32) +
                                  (1.0 - SIZE_SMOOTH_ALPHA) * prev_warp_size)

            warped, (w_used, h_used) = getWarp(frame, prev_corners.reshape(4,1,2), smooth_size=prev_warp_size)
            # crop slight border artifact trimming: threshold bright region -> bbox
            gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
            g = clahe.apply(gray_w)
            _, th_w = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            cnts, _ = cv2.findContours(th_w, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if cnts:
                best = max(cnts, key=cv2.contourArea)
                x,y,wc,hc = cv2.boundingRect(best)
                padx = max(1, int(0.01 * w_used)); pady = max(1, int(0.01 * h_used))
                x1 = max(0, x-padx); y1 = max(0, y-pady); x2 = min(w_used, x+wc+padx); y2 = min(h_used, y+hc+pady)
                cropped = warped[y1:y2, x1:x2]
            else:
                cropped = warped

            # hand off to enhancer
            with _worker_lock:
                _worker_input = (cropped.copy(), prev_corners.copy())

            # prefer enhanced if matches
            use_enhanced = False
            with _worker_lock:
                le = last_enhanced.copy() if last_enhanced is not None else None
                lec = last_enhanced_corners.copy() if last_enhanced_corners is not None else None
            if le is not None and lec is not None:
                if np.max(np.linalg.norm(lec - prev_corners, axis=1)) <= 8.0:
                    use_enhanced = True
            display = le if use_enhanced and le is not None else cropped
            label = "ENH" if use_enhanced else "LIVE"
            disp_small = display.copy()
            cv2.putText(disp_small, label, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
            top = cv2.resize(preview, (DISP_W, DISP_H))
            bottom = cv2.resize(disp_small, (DISP_W, DISP_H))
            combined = np.vstack([top, bottom])
            cv2.imshow("WorkFlow", combined)
            prev_warp = display.copy()
        else:
            top = cv2.resize(preview, (DISP_W, DISP_H))
            bottom = cv2.resize(prev_warp, (DISP_W, DISP_H)) if prev_warp is not None else top.copy()
            combined = np.vstack([top, bottom])
            cv2.imshow("WorkFlow", combined)

        # if KLT drifts too long, clear selected so detector must re-lock
        if lk_age > LK_MAX_AGE and candidate_age == 0:
            selected_corners = None
            prev_corners = None
            prev_warp_size = None
            tracker_points = None
            prev_gray = None
            selected_age = 0
            lk_age = 0
            candidate_age = 0
            candidate_corners = None
            bbox_tracker = None

        # debug mask windows (safe: don't query non-existent windows)
        if debug_mode:
            try:
                if 'mask_small' in locals() and mask_small is not None:
                    cv2.imshow("mask_small", cv2.resize(mask_small, (DISP_W, DISP_H)))
                if 'edges_small' in locals() and edges_small is not None:
                    cv2.imshow("edges_small", cv2.resize(edges_small, (DISP_W, DISP_H)))
            except Exception:
                # ignore any resize/show errors (e.g., if images absent)
                pass
        else:
            # destroy without querying window properties (avoid getWindowProperty)
            for wn in ("mask_small", "edges_small"):
                try:
                    cv2.destroyWindow(wn)
                except Exception:
                    # window may not exist — ignore
                    pass


        # keyboard
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('d'):
            debug_mode = not debug_mode
        elif key == ord('s'):
            if prev_warp is not None:
                fn = os.path.join(OUT_DIR, f"warp_{int(time.time())}.jpg")
                cv2.imwrite(fn, prev_warp)
                print("Saved:", fn)

finally:
    _worker_stop = True
    if USE_ENHANCER:
        worker.join(timeout=1.0)
    cap.release()
    cv2.destroyAllWindows()
    print("Exited cleanly")


Enhancer worker started
Enhancer worker stopping
Exited cleanly
